In [2]:
from venti.log import (
    LogFormat,
    LoggerConfig,
    LoggerManager,
    get_logger,
    log_performance,
    temporary_log_level,
)

In [3]:
import tempfile
import time
from pathlib import Path

# Simple logger

In [5]:
logger = get_logger("vlm_test", level="DEBUG", enable_file_logging=False)
logger.debug("Debug message - detailed information")
logger.info("Info message - general information")
logger.warning("Warning message - something to watch")
logger.error("Error message - something went wrong")
logger.critical("Critical message - urgent attention needed")

2025-07-21 14:55:40 | vlm_test             | INFO     | Info message - general information | taskName=Task-2 | message=Info message - general information | asctime=2025-07-21 14:55:40
2025-07-21 14:55:40 | vlm_test             | WARNING  | Warning message - something to watch | taskName=Task-2 | message=Warning message - something to watch | asctime=2025-07-21 14:55:40
2025-07-21 14:55:40 | vlm_test             | ERROR    | Error message - something went wrong | taskName=Task-2 | message=Error message - something went wrong | asctime=2025-07-21 14:55:40
2025-07-21 14:55:40 | vlm_test             | CRITICAL | Critical message - urgent attention needed | taskName=Task-2 | message=Critical message - urgent attention needed | asctime=2025-07-21 14:55:40


# Structured Logging

In [6]:
logger = get_logger("vlm_test", enable_file_logging=False)

# Log with extra context
logger.info("User performed action", extra={
    "user_id": "user_123",
    "action": "login",
    "ip_address": "192.168.1.1",
    "success": True,
    "timestamp": "2024-07-21T15:30:45"
})

2025-07-21 14:56:25 | vlm_test             | INFO     | User performed action | taskName=Task-2 | user_id=user_123 | action=login | ip_address=192.168.1.1 | success=True | message=User performed action | asctime=2025-07-21 14:56:25


# Performance Monitoring Test

In [7]:

logger = get_logger("vlm_test", enable_file_logging=False)

# Automatic performance tracking
with log_performance(logger, "data_processing"):
    time.sleep(0.5)  # Simulate some work
    logger.info("Processing 1000 records")

with log_performance(logger, "database_query"):
    time.sleep(0.2)  # Simulate database query
    logger.info("Query completed", extra={"rows_returned": 42})

2025-07-21 14:56:55 | vlm_test             | INFO     | Starting data_processing | taskName=Task-2 | message=Starting data_processing | asctime=2025-07-21 14:56:55
2025-07-21 14:56:56 | vlm_test             | INFO     | Processing 1000 records | taskName=Task-2 | message=Processing 1000 records | asctime=2025-07-21 14:56:56
2025-07-21 14:56:56 | vlm_test             | INFO     | data_processing completed in 0.502s | taskName=Task-2 | duration=0.5023913383483887 | operation=data_processing | message=data_processing completed in 0.502s | asctime=2025-07-21 14:56:56
2025-07-21 14:56:56 | vlm_test             | INFO     | Starting database_query | taskName=Task-2 | message=Starting database_query | asctime=2025-07-21 14:56:56
2025-07-21 14:56:56 | vlm_test             | INFO     | Query completed | taskName=Task-2 | rows_returned=42 | message=Query completed | asctime=2025-07-21 14:56:56
2025-07-21 14:56:56 | vlm_test             | INFO     | database_query completed in 0.201s | taskName=T

# Different Output Format

In [8]:
# Console format
console_logger = LoggerManager.get_logger("console", LoggerConfig(
    name="console", console_format=LogFormat.CONSOLE, enable_file_logging=False
))

# Detailed format
detailed_logger = LoggerManager.get_logger("detailed", LoggerConfig(
    name="detailed", console_format=LogFormat.DETAILED, enable_file_logging=False
))

# JSON format
json_logger = LoggerManager.get_logger("json", LoggerConfig(
    name="json", console_format=LogFormat.JSON, enable_file_logging=False
))

print("Console Format:")
console_logger.info("Sample message", extra={"user": "alice"})

print("\nDetailed Format:")
detailed_logger.info("Sample message", extra={"user": "bob"})

print("\nJSON Format:")
json_logger.info("Sample message", extra={"user": "charlie"})

Console Format:
2025-07-21 14:58:52 | console              | INFO     | Sample message | taskName=Task-2 | user=alice | message=Sample message | asctime=2025-07-21 14:58:52

Detailed Format:
2025-07-21 14:58:52 | detailed | INFO | /tmp/ipykernel_124417/785116359.py:20 | <module> | PID:124417 | Thread:140053702707008 | Sample message | taskName=Task-2 | user=bob | message=Sample message | asctime=2025-07-21 14:58:52

JSON Format:
{"timestamp": "2025-07-21T14:58:52.682006", "level": "INFO", "logger": "json", "message": "Sample message", "module": "785116359", "function": "<module>", "line": 23, "process_id": 124417, "thread_id": 140053702707008, "taskName": "Task-2", "user": "charlie"}


# Context Managers 

In [10]:
logger = get_logger("vlm_test", level="INFO", enable_file_logging=False)

print("Normal log level (INFO):")
logger.debug("This won't show")
logger.info("This will show")

print("\nTemporary DEBUG level:")
with temporary_log_level(logger, "DEBUG"):
    logger.debug("Now this will show!")
    logger.info("This still shows")

print("\nBack to normal:")
logger.debug("Hidden again")
logger.info("Still shows")

Normal log level (INFO):
2025-07-21 15:00:23 | vlm_test             | INFO     | This will show | taskName=Task-2 | message=This will show | asctime=2025-07-21 15:00:23

Temporary DEBUG level:
2025-07-21 15:00:23 | vlm_test             | INFO     | This still shows | taskName=Task-2 | message=This still shows | asctime=2025-07-21 15:00:23

Back to normal:
2025-07-21 15:00:23 | vlm_test             | INFO     | Still shows | taskName=Task-2 | message=Still shows | asctime=2025-07-21 15:00:23


# Error Handling

In [11]:
logger = get_logger("error_test", enable_file_logging=False)

# Test exception logging
try:
    result = 10 / 0
except Exception as e:
    logger.error("Division by zero occurred", extra={
        "operation": "division",
        "numerator": 10,
        "denominator": 0,
        "error_type": type(e).__name__
    }, exc_info=True)

try:
    data = {"name": "Alice"}
    age = data["age"]  # KeyError
except Exception as e:
    logger.error("Key not found", extra={
        "key": "age",
        "available_keys": list(data.keys()),
        "error_type": type(e).__name__
    }, exc_info=True)

2025-07-21 15:01:02 | error_test           | ERROR    | Division by zero occurred
Traceback (most recent call last):
  File "/tmp/ipykernel_124417/1161170279.py", line 5, in <module>
    result = 10 / 0
             ~~~^~~
ZeroDivisionError: division by zero | taskName=Task-2 | operation=division | numerator=10 | denominator=0 | error_type=ZeroDivisionError | message=Division by zero occurred | asctime=2025-07-21 15:01:02
2025-07-21 15:01:02 | error_test           | ERROR    | Key not found
Traceback (most recent call last):
  File "/tmp/ipykernel_124417/1161170279.py", line 16, in <module>
    age = data["age"]  # KeyError
          ~~~~^^^^^^^
KeyError: 'age' | taskName=Task-2 | key=age | available_keys=['name'] | error_type=KeyError | message=Key not found | asctime=2025-07-21 15:01:02


# Real-World Simulation

In [13]:
import random

app_logger = get_logger("webapp", enable_file_logging=False)

# Simulate HTTP requests
requests = [
    {"method": "GET", "path": "/api/users", "status": 200},
    {"method": "POST", "path": "/api/orders", "status": 201},
    {"method": "GET", "path": "/api/orders/123", "status": 404},
    {"method": "PUT", "path": "/api/users/456", "status": 500},
]

for i, req in enumerate(requests):
    request_id = f"req_{i+1:03d}"
    duration = random.uniform(0.05, 0.3)

    # Log request
    app_logger.info("HTTP Request", extra={
        "request_id": request_id,
        "method": req["method"],
        "path": req["path"],
        "user_agent": "Mozilla/5.0 (Test Browser)",
        "ip": "192.168.1.100"
    })

    time.sleep(duration)

    # Log response
    level = "error" if req["status"] >= 400 else "info"
    getattr(app_logger, level)("HTTP Response", extra={
        "request_id": request_id,
        "status_code": req["status"],
        "duration_ms": duration * 1000,
        "success": req["status"] < 400
    })


2025-07-21 15:03:36 | webapp               | INFO     | HTTP Request | taskName=Task-2 | request_id=req_001 | method=GET | path=/api/users | user_agent=Mozilla/5.0 (Test Browser) | ip=192.168.1.100 | message=HTTP Request | asctime=2025-07-21 15:03:36
2025-07-21 15:03:36 | webapp               | INFO     | HTTP Response | taskName=Task-2 | request_id=req_001 | status_code=200 | duration_ms=82.61936814889013 | success=True | message=HTTP Response | asctime=2025-07-21 15:03:36
2025-07-21 15:03:36 | webapp               | INFO     | HTTP Request | taskName=Task-2 | request_id=req_002 | method=POST | path=/api/orders | user_agent=Mozilla/5.0 (Test Browser) | ip=192.168.1.100 | message=HTTP Request | asctime=2025-07-21 15:03:36
2025-07-21 15:03:36 | webapp               | INFO     | HTTP Response | taskName=Task-2 | request_id=req_002 | status_code=201 | duration_ms=90.7975668882739 | success=True | message=HTTP Response | asctime=2025-07-21 15:03:36
2025-07-21 15:03:36 | webapp             

# File Logging

In [18]:
# Create temporary directory
temp_dir = tempfile.mkdtemp()
print(f"Log files will be created in: {temp_dir}")

# Create file logger
file_logger = get_logger(
    name="file_test",
    level="DEBUG",
    enable_file_logging=True,
    enable_json_logging=True,
    log_dir=temp_dir
)

# Generate some logs
for i in range(10):
    file_logger.info(f"File log message {i}", extra={
        "iteration": i,
        "batch": i // 3,
        "data": f"sample_data_{i}"
    })

if i % 3 == 0:
    file_logger.warning(f"Batch {i//3} checkpoint")

# Show created files
log_files = list(Path(temp_dir).glob("*"))
print(f"\nCreated {len(log_files)} files:")
for file in sorted(log_files):
    print(f"  - {file.name} ({file.stat().st_size} bytes)")

# Show sample content
main_log = Path(temp_dir) / "file_test.log"
if main_log.exists():
    print(f"\nSample content from {main_log.name}:")
    with open(main_log) as f:
        lines = f.readlines()[:3]  # Show first 3 lines
        for line in lines:
            print(f"  {line.strip()}")

print(f"\nCheck the files in: {temp_dir}")

Log files will be created in: /tmp/tmp_cyul1ao
2025-07-21 15:05:38 | file_test            | INFO     | File log message 0 | taskName=Task-2 | iteration=0 | batch=0 | data=sample_data_0 | message=File log message 0 | asctime=2025-07-21 15:05:38
2025-07-21 15:05:38 | file_test            | INFO     | File log message 1 | taskName=Task-2 | iteration=1 | batch=0 | data=sample_data_1 | message=File log message 1 | asctime=2025-07-21 15:05:38
2025-07-21 15:05:38 | file_test            | INFO     | File log message 2 | taskName=Task-2 | iteration=2 | batch=0 | data=sample_data_2 | message=File log message 2 | asctime=2025-07-21 15:05:38
2025-07-21 15:05:38 | file_test            | INFO     | File log message 3 | taskName=Task-2 | iteration=3 | batch=1 | data=sample_data_3 | message=File log message 3 | asctime=2025-07-21 15:05:38
2025-07-21 15:05:38 | file_test            | INFO     | File log message 4 | taskName=Task-2 | iteration=4 | batch=1 | data=sample_data_4 | message=File log message 

# Performance Benchmark

In [19]:
logger = get_logger("benchmark", enable_file_logging=False)

# Measure logging performance
message_count = 1000
start_time = time.time()

for i in range(message_count):
    logger.info(f"Benchmark message {i}", extra={
        "iteration": i,
        "test": "performance"
    })

end_time = time.time()
duration = end_time - start_time
rate = message_count / duration

print("Performance Results:")
print(f"  Messages: {message_count}")
print(f"  Duration: {duration:.3f} seconds")
print(f"  Rate: {rate:.0f} messages/second")

2025-07-21 15:06:26 | benchmark            | INFO     | Benchmark message 0 | taskName=Task-2 | iteration=0 | test=performance | message=Benchmark message 0 | asctime=2025-07-21 15:06:26
2025-07-21 15:06:26 | benchmark            | INFO     | Benchmark message 1 | taskName=Task-2 | iteration=1 | test=performance | message=Benchmark message 1 | asctime=2025-07-21 15:06:26
2025-07-21 15:06:26 | benchmark            | INFO     | Benchmark message 2 | taskName=Task-2 | iteration=2 | test=performance | message=Benchmark message 2 | asctime=2025-07-21 15:06:26
2025-07-21 15:06:26 | benchmark            | INFO     | Benchmark message 3 | taskName=Task-2 | iteration=3 | test=performance | message=Benchmark message 3 | asctime=2025-07-21 15:06:26
2025-07-21 15:06:26 | benchmark            | INFO     | Benchmark message 4 | taskName=Task-2 | iteration=4 | test=performance | message=Benchmark message 4 | asctime=2025-07-21 15:06:26
2025-07-21 15:06:26 | benchmark            | INFO     | Benchmark

# Interactive

In [21]:
# Create different loggers for testing
basic = get_logger("basic", level="DEBUG", enable_file_logging=False)
json_logger = get_logger("json", level="INFO", enable_file_logging=False)

print("Loggers created! Try these commands:")
print("basic.info('Hello world!')")
print("basic.info('User action', extra={'user_id': '123', 'action': 'login'})")
print("with log_performance(basic, 'test'): time.sleep(0.1)")
print("with temporary_log_level(basic, 'DEBUG'): basic.debug('Debug message')")

Loggers created! Try these commands:
basic.info('Hello world!')
basic.info('User action', extra={'user_id': '123', 'action': 'login'})
with log_performance(basic, 'test'): time.sleep(0.1)
with temporary_log_level(basic, 'DEBUG'): basic.debug('Debug message')


In [22]:
with log_performance(basic, 'test'): time.sleep(0.1)

2025-07-21 15:07:58 | basic                | INFO     | Starting test | taskName=Task-2 | message=Starting test | asctime=2025-07-21 15:07:58
2025-07-21 15:07:58 | basic                | INFO     | test completed in 0.101s | taskName=Task-2 | duration=0.10098147392272949 | operation=test | message=test completed in 0.101s | asctime=2025-07-21 15:07:58
